# BANA 4373 — Homework 3  
## Building a WOWI Economic Data Agent

**Course:** ECON 4370 – Applied Data Tools for Economics and Business  
**Instructor:** Dr. Fidel Gonzalez  
**Assignment:** Homework 3  
**Topic:** AI Agents and Economic Data  

---

# Overview

In this assignment you will build a simple **economic data agent** that can answer questions using real economic datasets.

The agent will follow a framework we introduced in class called the **WOWI framework**:

### WOWI Framework

**WHAT → OBTAIN → WORK → INTERPRET**

| Step | Meaning |
|-----|------|
| **WHAT** | Understand the user’s question |
| **OBTAIN** | Retrieve the necessary data |
| **WORK** | Analyze the data using tools |
| **INTERPRET** | Explain the economic meaning |

This framework mirrors the workflow used by economists and data analysts:

**Question → Data → Analysis → Interpretation**

Your agent will retrieve data from real sources such as:

- **FRED (Federal Reserve Economic Data)**
- **U.S. Census American Community Survey (ACS)**

---

# Learning Objectives

By completing this assignment you will learn how to:

- Use **APIs** to retrieve real-world economic data
- Explore and identify relevant **economic data series**
- Build simple **data analysis tools**
- Understand how **AI agents interact with tools**
- Interpret economic results like an applied economist

---

# What you must do

### Part A — API keys

You must obtain and use:

- an **OpenAI API key**
- a **FRED API key**
- a **Census API key**

These keys allow your agent to:
- interpret user questions
- retrieve macroeconomic data
- retrieve socioeconomic data

---

### Part B — Data dictionaries

You will start with:

- **2 example FRED series**
- **2 example ACS variables**

You must add:

- **2 more FRED series**
- **2 more ACS variables**

So your final dictionaries must contain **at least 4 series or variables from each source**.

⚠️ Important:  
The example variables provided in the notebook **may be used in your analysis**, but they **do not count toward your required additions**.

---

### Part C — Tools

You are given **three starter tools**:

1. A **FRED data retrieval tool**
2. A **Census ACS data retrieval tool**
3. A **plotting tool**

You must build **two additional tools** of your own.

These tools should help your agent **analyze or transform the data**.

Examples include:

- correlation
- moving averages
- summary statistics
- ranking states
- differences between variables

An example tool will be shown in the notebook for guidance, but **you may not use that example as one of your required tools**.

---

### Part D — Agents

You will work with **two types of workflows**:

1. A **scripted example** from class
2. A **WOWI agent**

The scripted example follows fixed instructions.

The WOWI agent is more flexible because it:

- interprets the question
- decides what data to obtain
- selects tools
- produces an answer

---

### Part E — Demo runs

You must complete **10 demo runs** using your **WOWI agent**.

Across your demo runs you must:

- use **both data sources** (FRED and ACS)
- use **both of your custom tools**
- write a short interpretation of the results

Each interpretation should explain:

- what the agent chose to do
- what the output shows
- what the result means economically

---

# What you must turn in

You must submit **one completed Jupyter Notebook (.ipynb file)**.

Before submitting, make sure that:

- all required sections are completed
- your two tools are implemented
- all **10 demo runs** are included
- each run includes an **interpretation**
- the notebook runs from top to bottom without major errors

Upload the notebook to **Blackboard**.

---

# Important note

This notebook contains:

- starter code
- example sections
- **TODO sections** that you must complete

Work carefully through each section. The assignment is designed to be **challenging but very manageable if you proceed step by step**.


# 1) API Keys

You must obtain **three API keys**.

## OpenAI
Use the same key you used for the in-class AI agent activity.

## FRED
Create a FRED API key here:

- FRED API documentation and key request page:  
  `https://fred.stlouisfed.org/docs/api/api_key.html`

## Census
Request a Census API key here:

- Census API key signup page:  
  `https://api.census.gov/data/key_signup.html`

---

## Why this matters

In real applied work, analysts rarely type data by hand.  
Instead, they use APIs to retrieve data automatically.

That is one of the major themes of this course:
> **Applied economists and business analysts must know how to obtain data programmatically.**

---

## Instructions

Run the next cell and paste your keys when prompted.

- Your input will be hidden.
- Do **not** hard-code your keys into the notebook.
- Do **not** upload a notebook that visibly contains your keys.


In [ ]:
from getpass import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.environ.get("FRED_API_KEY"):
    os.environ["FRED_API_KEY"] = getpass("Enter your FRED API key: ")

if not os.environ.get("CENSUS_API_KEY"):
    os.environ["CENSUS_API_KEY"] = getpass("Enter your Census API key: ")

print("OpenAI key loaded:", "YES" if os.environ.get("OPENAI_API_KEY") else "NO")
print("FRED key loaded:", "YES" if os.environ.get("FRED_API_KEY") else "NO")
print("Census key loaded:", "YES" if os.environ.get("CENSUS_API_KEY") else "NO")



# 2) Imports and Setup

Run the next cell. If you are missing any packages, uncomment the installation line.


In [ ]:
# If needed, uncomment this line:
# !pip -q install pandas numpy plotly requests openai fredapi statsmodels

import json
import re
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf

from openai import OpenAI
from fredapi import Fred



# 3) Build Your Data Dictionaries

This section is **very important**.

You will use **two dictionaries**:
- one for **FRED series**
- one for **ACS variables**

You are given **2 starter entries** in each dictionary.  
You must add **2 more** to each.

---

## FRED dictionary

The two starter examples are:

- `inflation` → CPIAUCSL  
- `unemployment` → UNRATE  

You must add **at least 2 more** FRED series.

Examples you might consider:
- federal funds rate
- oil price
- GDP
- industrial production
- consumer sentiment
- mortgage rates
- wages

---

## ACS dictionary

The two starter examples are:

- `median_income` → B19013_001E  
- `population` → B01003_001E  

You must add **at least 2 more** ACS variables.

Examples you might consider:
- median home value
- poverty count
- rent
- unemployment-related measures
- education measures

---

## Where to search

### FRED
Search on the FRED website for series IDs.

### ACS variables
Use the ACS variable search pages or API documentation.

---

## Your task

Complete the two dictionaries below.

### Requirement
By the time you are done:
- `FRED_SERIES` must have **at least 4 entries**
- `ACS_VARIABLES` must have **at least 4 entries**


In [ ]:
# TODO: Add at least 2 more FRED series
FRED_SERIES = {
    "inflation": "CPIAUCSL",
    "unemployment": "UNRATE",

    # TODO: add at least 2 more below
    # "fed_funds_rate": "FEDFUNDS",
    # "oil_price": "DCOILWTICO",
}

# TODO: Add at least 2 more ACS variables
ACS_VARIABLES = {
    "median_income": "B19013_001E",
    "population": "B01003_001E",

    # TODO: add at least 2 more below
    # "median_home_value": "B25077_001E",
    # "poverty_count": "B17001_002E",
}

print("FRED entries:", len(FRED_SERIES))
print("ACS entries:", len(ACS_VARIABLES))

FRED_SERIES, ACS_VARIABLES



## 3.1) Describe Your Choices

Briefly explain:
- which **2 new FRED series** you added and why,
- which **2 new ACS variables** you added and why.

This helps show that you explored the data sources intentionally.


### Your explanation

**New FRED series I added:**
- 
- 

**Why I chose them:**
- 

**New ACS variables I added:**
- 
- 

**Why I chose them:**
- 



# 4) Starter Tool 1 — FRED Data Tool

This tool retrieves a FRED series by series ID.

It returns a `pandas.Series` with a datetime index.

---

## What this tool does
- connects to the FRED API
- pulls one series
- converts the index to dates
- labels the series with the series ID

Run the next cell.


In [ ]:
fred = Fred(api_key=os.environ.get("FRED_API_KEY"))

def get_fred_series(series_id, start="1990-01-01", end=None):
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s.index = pd.to_datetime(s.index)
    s.name = series_id
    return s



# 5) Starter Tool 2 — ACS Data Tool

This tool retrieves a single ACS variable for **all states** for a given year.

For example, if you request:
- `B19013_001E` for year `2022`

you will get state-level median household income for that year.

---

## Important note

FRED is usually **time-series** data.  
ACS is often **cross-sectional** data in this notebook:
- one year
- many states

That means the plotting and interpretation for ACS may look different from FRED.

Run the next cell.


In [ ]:
def get_acs_state_data(variable, year=2022):
    api_key = os.environ.get("CENSUS_API_KEY")
    url = f"https://api.census.gov/data/{year}/acs/acs1"
    params = {
        "get": f"NAME,{variable}",
        "for": "state:*",
        "key": api_key
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data[1:], columns=data[0])
    df[variable] = pd.to_numeric(df[variable], errors="coerce")
    return df.sort_values(variable, ascending=False).reset_index(drop=True)



# 6) Starter Tool 3 — Plotting Tool

This is your basic plotting tool.

It can handle two common cases:

## Case A — FRED
If you give it time-series data, it makes a **line chart**.

## Case B — ACS
If you give it a state-level ACS dataset, it can make a **bar chart**.

Run the next cell.


In [ ]:
def plot_data(df, x, y, title, kind="line"):
    if kind == "line":
        fig = px.line(df, x=x, y=y, title=title)
    elif kind == "bar":
        fig = px.bar(df, x=x, y=y, title=title)
    elif kind == "scatter":
        fig = px.scatter(df, x=x, y=y, title=title)
    else:
        raise ValueError("kind must be 'line', 'bar', or 'scatter'")
    fig.show()



# 7) Helpful Utility Functions

These are not counted as your two required custom tools.  
They are helper functions you may use in your workflow.

Run the next cell.


In [ ]:
def to_monthly(series):
    """Convert a higher-frequency series to monthly data if needed."""
    s = series.dropna().copy()
    if len(s) >= 3:
        gap = np.median(np.diff(s.index.values).astype("timedelta64[D]").astype(int))
    else:
        gap = 31

    if gap < 20:
        s = s.resample("M").mean()
    else:
        s = s.resample("M").last()

    s.index = s.index.to_period("M").to_timestamp("M")
    return s

def yoy_pct(series):
    """Year-over-year percent change."""
    return 100 * (series / series.shift(12) - 1)

def build_fred_dataset(series_ids, start="1990-01-01"):
    frames = []
    for sid in series_ids:
        s = get_fred_series(sid, start=start)
        s = to_monthly(s)
        frames.append(s)
    df = pd.concat(frames, axis=1)
    df.index.name = "date"
    return df



# 8) Scripted Agent Example from Class  
## Example only — do not use this as one of your 10 demo runs

This cell shows the idea of a **scripted agent** or **hard-coded workflow**.

It does not really "reason" about the question.  
Instead, the programmer has already decided:
- which variables to use,
- which transformations to compute,
- which plot to make.

That is why this is closer to a **script** than a flexible agent.

Run the next example and observe what it does.


In [ ]:
# Example from class (DO NOT count this as one of your 10 demo runs)

example_df = build_fred_dataset(["CPIAUCSL", "UNRATE"], start="1990-01-01")
example_df["inflation_yoy"] = yoy_pct(example_df["CPIAUCSL"])

plot_data(
    example_df.reset_index(),
    x="date",
    y=["inflation_yoy", "UNRATE"],
    title="Example scripted workflow: inflation and unemployment since 1990",
    kind="line"
)

example_df.tail()



## 8.1) Short reflection on the scripted example

In **2–4 sentences**, explain why the example above is better described as a **scripted workflow** than as a fully flexible AI agent.


### Your reflection

Write here.



# 9) WOW Agent Architecture

Now you will work with a more flexible structure:

# **WHAT → OBTAIN → WORK**

## WHAT
Interpret the question:
- Which data source should be used?
- Which concepts are being requested?
- What kind of task is needed?

## OBTAIN
Retrieve the data:
- FRED data
- ACS data

## WORK
Analyze the data:
- plot
- correlation
- summary
- regression
- your two new tools

---

## Important design choice

To keep the assignment manageable, the planner in this notebook will be **simple**.

It will return a small JSON object such as:

```json
{
  "source": "fred",
  "concepts": ["inflation", "unemployment"],
  "task": "compare",
  "start_year": 1990,
  "year": null
}
```

or

```json
{
  "source": "acs",
  "concepts": ["median_income", "population"],
  "task": "compare",
  "start_year": null,
  "year": 2022
}
```



# 10) Simple Planner

This is the **WHAT** stage.

The planner uses the OpenAI API to turn a natural-language question into a small JSON plan.

### Output fields
- `source`: `"fred"` or `"acs"`
- `concepts`: list of dictionary keys
- `task`: what kind of work should be done
- `start_year`: mainly for FRED
- `year`: mainly for ACS

Run the next cell.


In [ ]:
client = OpenAI()

PLANNER_PROMPT = '''
You are helping a student build a simple WOW economic data agent.

Use only the approved dictionary keys that the student provides in the notebook:
- FRED keys come from FRED_SERIES
- ACS keys come from ACS_VARIABLES

Return ONLY valid JSON with this exact structure:
{
  "source": "fred" or "acs",
  "concepts": ["concept1", "concept2"],
  "task": "compare" or "describe" or "regression" or "custom_tool",
  "start_year": 1990,
  "year": 2022
}

Rules:
- Use "fred" for macroeconomic time-series questions.
- Use "acs" for cross-state socioeconomic questions.
- Use only concept names that exist in the notebook dictionaries.
- If the user asks for trends over time in FRED, include a start_year.
- If the user asks about ACS, set a year such as 2022 unless the user provides a different year.
- If the question asks for a relationship like Phillips Curve, use task = "regression".
- If the question asks to use one of the student's custom tools, use task = "custom_tool".
- Return only JSON.
'''

def get_plan(question, model="gpt-4.1-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": question}
        ],
        temperature=0
    )
    text = response.choices[0].message.content
    return json.loads(text)



# 11) Build Two New Tools  
## Example tool below is for inspiration only — you may NOT use it as one of your two required tools

Below is an example of a simple analysis tool:

```python
def growth_rate(series):
    return series.pct_change() * 100
```

This is a good example of the level of difficulty expected.

### But:
You may **not** submit `growth_rate()` as one of your two required tools.

---

## What your two tools should look like

Your tools should:
- accept a `Series` or `DataFrame`,
- do something useful,
- return a result,
- print something informative.

### Good examples
- moving average
- correlation between two columns
- summary statistics
- difference between two variables
- rank top states
- z-score transformation
- min/max finder

You may create tools inspired by these examples, but write your own.

---

## Your task

Create **Tool 4** and **Tool 5** below.

Scaffolding is provided.


In [ ]:
# TODO: Create your own Tool 4
def tool4(df_or_series, *args, **kwargs):
    '''
    Describe what your tool does here.

    Input:
    - explain the expected input

    Output:
    - explain what it returns
    '''

    # TODO: write your code here
    pass


In [ ]:
# TODO: Create your own Tool 5
def tool5(df_or_series, *args, **kwargs):
    '''
    Describe what your tool does here.

    Input:
    - explain the expected input

    Output:
    - explain what it returns
    '''

    # TODO: write your code here
    pass



## 11.1) Explain your two tools

Briefly explain:
- what Tool 4 does,
- what Tool 5 does,
- why they are useful.

This explanation is part of the assignment.


### Your explanation

**Tool 4:**
- 

**Why it is useful:**
- 

**Tool 5:**
- 

**Why it is useful:**
- 



# 12) WOW Agent Runner

This function will carry out the **OBTAIN** and **WORK** stages.

It will:
1. call the planner,
2. decide whether to use FRED or ACS,
3. retrieve the needed data,
4. perform a basic action depending on the task.

You may modify this function if needed.

Read it carefully before using it.


In [ ]:
def run_wow_agent(question):
    plan = get_plan(question)

    print("PLAN")
    print(json.dumps(plan, indent=2))

    source = plan["source"]
    concepts = plan["concepts"]
    task = plan["task"]

    # ----- FRED path -----
    if source == "fred":
        series_ids = [FRED_SERIES[c] for c in concepts]
        start = f'{plan.get("start_year", 1990)}-01-01'
        df = build_fred_dataset(series_ids, start=start)

        # Helpful default transformations
        if "CPIAUCSL" in df.columns:
            df["inflation_yoy"] = yoy_pct(df["CPIAUCSL"])
        if "UNRATE" in df.columns:
            pass

        # Basic compare/describe plot
        if task in ["compare", "describe"]:
            plot_cols = []
            for c in concepts:
                sid = FRED_SERIES[c]
                if sid == "CPIAUCSL" and "inflation_yoy" in df.columns:
                    plot_cols.append("inflation_yoy")
                else:
                    plot_cols.append(sid)

            plot_data(
                df.reset_index(),
                x="date",
                y=plot_cols,
                title=question,
                kind="line"
            )
            return {"plan": plan, "data": df.tail(12)}

        # Simple regression example
        elif task == "regression":
            if "CPIAUCSL" in df.columns and "UNRATE" in df.columns:
                reg_df = df.copy()
                reg_df["inflation_yoy"] = yoy_pct(reg_df["CPIAUCSL"])
                reg_df = reg_df[["inflation_yoy", "UNRATE"]].dropna()

                model = smf.ols("inflation_yoy ~ UNRATE", data=reg_df).fit()
                print(model.summary())
                return {
                    "plan": plan,
                    "params": model.params.to_dict(),
                    "r2": model.rsquared
                }
            else:
                print("Default regression requires CPIAUCSL and UNRATE.")
                return {"plan": plan}

        elif task == "custom_tool":
            print("Use one of your custom tools below after inspecting the dataset.")
            return {"plan": plan, "data": df}

        else:
            print("Unknown FRED task.")
            return {"plan": plan, "data": df}

    # ----- ACS path -----
    elif source == "acs":
        year = plan.get("year", 2022)
        variables = [ACS_VARIABLES[c] for c in concepts]

        if len(variables) == 1:
            var = variables[0]
            df = get_acs_state_data(var, year=year)
            plot_data(
                df.head(15),
                x="NAME",
                y=var,
                title=f"{concepts[0]} across states ({year})",
                kind="bar"
            )
            return {"plan": plan, "data": df.head(10)}

        elif len(variables) >= 2:
            var1, var2 = variables[:2]
            df1 = get_acs_state_data(var1, year=year)[["NAME", var1]]
            df2 = get_acs_state_data(var2, year=year)[["NAME", var2]]
            merged = df1.merge(df2, on="NAME", how="inner")

            if task in ["compare", "describe"]:
                plot_data(
                    merged,
                    x=var1,
                    y=var2,
                    title=f"{concepts[0]} vs {concepts[1]} across states ({year})",
                    kind="scatter"
                )
                return {"plan": plan, "data": merged.head(10)}

            elif task == "custom_tool":
                print("Use one of your custom tools below after inspecting the ACS dataset.")
                return {"plan": plan, "data": merged}

            else:
                return {"plan": plan, "data": merged}

    else:
        print("Unknown source.")
        return {"plan": plan}



# 13) Example WOW Agent Demo Run  
## Example only — do not count this as one of your 10 demo runs

This is a starter example to help you see how the WOW agent works.

Run the cell below and examine:
- the plan
- the data source chosen
- the output

You may use this as a model for your own demo runs, but **you may not count this exact example as one of your 10 required runs**.


In [ ]:
example_run = run_wow_agent("Compare inflation and unemployment since 1990")
example_run



# 14) Example Interpretation  
## Example only — do not copy this as one of your graded interpretations

Below is the **style** of interpretation expected.

> The agent selected FRED as the data source and chose inflation and unemployment as the relevant concepts. It retrieved CPI and unemployment data starting in 1990 and plotted the transformed inflation series against the unemployment rate. The graph suggests that the relationship between inflation and unemployment changes over time and is not perfectly stable. This shows why economic relationships can be more complicated than a simple textbook rule.

Your own interpretations do **not** need to match this exactly, but they should be:
- clear,
- concise,
- economically meaningful.



# 15) Your 10 Required Demo Runs

You must complete **10 demo runs** using your WOW agent.

## Requirements
Across your 10 runs:
- use **both data sources** at least once:
  - FRED
  - ACS
- use **both of your custom tools** at least once
- include a mix of economic questions

---

## Good prompt ideas

### FRED-style prompts
- Compare inflation and unemployment since 1990
- Compare inflation and the federal funds rate since 2000
- Compare oil prices and inflation after 2015
- Compare wages and inflation after 2020
- Is there evidence of a Phillips Curve since 1990?

### ACS-style prompts
- Compare median income and population across states in 2022
- Compare median income and home values across states
- Compare poverty and median income across states
- Describe the distribution of one ACS variable across states
- Use one of your custom tools on an ACS dataset

---

## Important
Do **not** just run 10 cells without thinking.  
Each run should be followed by a **3–5 sentence interpretation in your own words**.


## Demo Run 1

In [ ]:
# TODO: Replace this example prompt with your own
question_1 = "Replace this with your question"
result_1 = run_wow_agent(question_1)
result_1


### Interpretation for Demo Run 1

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 2

In [ ]:
# TODO: Replace this example prompt with your own
question_2 = "Replace this with your question"
result_2 = run_wow_agent(question_2)
result_2


### Interpretation for Demo Run 2

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 3

In [ ]:
# TODO: Replace this example prompt with your own
question_3 = "Replace this with your question"
result_3 = run_wow_agent(question_3)
result_3


### Interpretation for Demo Run 3

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 4

In [ ]:
# TODO: Replace this example prompt with your own
question_4 = "Replace this with your question"
result_4 = run_wow_agent(question_4)
result_4


### Interpretation for Demo Run 4

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 5

In [ ]:
# TODO: Replace this example prompt with your own
question_5 = "Replace this with your question"
result_5 = run_wow_agent(question_5)
result_5


### Interpretation for Demo Run 5

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 6

In [ ]:
# TODO: Replace this example prompt with your own
question_6 = "Replace this with your question"
result_6 = run_wow_agent(question_6)
result_6


### Interpretation for Demo Run 6

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 7

In [ ]:
# TODO: Replace this example prompt with your own
question_7 = "Replace this with your question"
result_7 = run_wow_agent(question_7)
result_7


### Interpretation for Demo Run 7

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 8

In [ ]:
# TODO: Replace this example prompt with your own
question_8 = "Replace this with your question"
result_8 = run_wow_agent(question_8)
result_8


### Interpretation for Demo Run 8

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 9

In [ ]:
# TODO: Replace this example prompt with your own
question_9 = "Replace this with your question"
result_9 = run_wow_agent(question_9)
result_9


### Interpretation for Demo Run 9

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?


## Demo Run 10

In [ ]:
# TODO: Replace this example prompt with your own
question_10 = "Replace this with your question"
result_10 = run_wow_agent(question_10)
result_10


### Interpretation for Demo Run 10

Write 3–5 sentences here.

- What source did the agent choose?
- What variables did it use?
- What did the output show?
- What is your economic interpretation?



# 16) Use Your Custom Tools

In this section, demonstrate your two custom tools clearly.

You must show:
- at least one use of **Tool 4**
- at least one use of **Tool 5**

These can be based on one of your demo run datasets or a new dataset.

If one of your 10 demo runs already used the tool clearly, you may reference that here and show it again briefly.


In [ ]:
# TODO: Demonstrate Tool 4 here

# Example pattern:
# df = build_fred_dataset(["CPIAUCSL", "UNRATE"], start="2000-01-01")
# result_tool4 = tool4(df, ...)
# result_tool4


### Interpretation for Tool 4

Write 2–4 sentences here explaining what Tool 4 showed.


In [ ]:
# TODO: Demonstrate Tool 5 here

# Example pattern:
# df = get_acs_state_data("B19013_001E", year=2022)
# result_tool5 = tool5(df, ...)
# result_tool5


### Interpretation for Tool 5

Write 2–4 sentences here explaining what Tool 5 showed.



# 17) Final Reflection

Answer the following questions thoughtfully.

### 1. API keys
What challenges, if any, did you encounter obtaining or using the OpenAI, FRED, and Census API keys?

### 2. Data discovery
How did you find your additional FRED series and ACS variables?

### 3. Tools
What do your two new tools do, and why are they useful?

### 4. Agents
What is the difference between the **scripted example** and the **WOW agent**?

### 5. Interpretation
What did you learn from doing 10 demo runs instead of just one or two?


### Your final reflection

**1. API keys**  
Write here.

**2. Data discovery**  
Write here.

**3. Tools**  
Write here.

**4. Scripted agent vs WOW agent**  
Write here.

**5. What I learned from doing 10 demo runs**  
Write here.



# 18) Suggested Grading Rubric

| Component | Points |
|---|---:|
| API keys set up correctly | 10 |
| FRED dictionary expanded to 4+ entries | 10 |
| ACS dictionary expanded to 4+ entries | 10 |
| Scripted-agent reflection | 5 |
| Tool 4 completed | 10 |
| Tool 5 completed | 10 |
| WOW agent works and is used correctly | 15 |
| 10 demo runs completed | 15 |
| Interpretations are clear and thoughtful | 10 |
| Final reflection | 5 |

**Total = 100 points**



# 19) Final Checklist Before You Submit

Before submitting, make sure:

- [ ] I entered and used all three API keys  
- [ ] I added at least 2 new FRED series  
- [ ] I added at least 2 new ACS variables  
- [ ] I ran the scripted example and wrote the reflection  
- [ ] I created **two** new tools  
- [ ] I completed **10** demo runs  
- [ ] I used both data sources at least once  
- [ ] I demonstrated both of my custom tools  
- [ ] I wrote interpretations in my own words  
- [ ] My notebook runs from top to bottom without major errors  

---

## Closing note

This homework is meant to feel like a real applied data project:
- retrieve data,
- build tools,
- use an agent,
- interpret economic results.

Take it step by step. If you do that, you will be surprised by how much you can build.
